# Prediksi Rute Pengiriman Efisien — Indonesia (Dataset & EDA)

**Tujuan:** memprediksi / merencanakan rute pengiriman yang paling efisien.

**Dataset (asli Indonesia — ditemukan di Kaggle):** [Last Mile Delivery](https://www.kaggle.com/datasets/meissydanjiah/last-mile-delivery) — koordinat pelanggan *end-mile delivery* (EMD) dari 6 kecamatan di Kota Pontianak, Kalimantan Barat. Data dikumpulkan dengan *proportionate stratified random sampling* berdasarkan 9 strata jalan (dari jalan utama hingga jalan pendukung). Berisi: nama kecamatan, nama jalan per strata, jarak (meter), bagian, dan koordinat lat/long.

**Catatan:** dataset Hugging Face untuk rute pengiriman Indonesia tidak ditemukan (hanya korpus NLP yang tidak relevan). Dataset ini adalah pengganti nyata untuk eksplorasi rute last-mile.

**Sumber data:** meissydanjiah (2024), *LAST MILE DELIVERY*, Kaggle — https://www.kaggle.com/datasets/meissydanjiah/last-mile-delivery

In [ ]:
# Unduh dataset asli Indonesia: Last Mile Delivery (Pontianak, Kalimantan Barat)
# Sumber: https://www.kaggle.com/datasets/meissydanjiah/last-mile-delivery
import kagglehub
import pandas as pd
import os, glob, shutil

path = kagglehub.dataset_download("meissydanjiah/last-mile-delivery")
print("Cache:", path)

csv_path = glob.glob(os.path.join(path, "*.csv"))[0]

# File ini memakai pemisah titik-koma (;) dengan pasangan koordinat "Latitude,Longitude"
df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

# Salin ke /home/user agar tersimpan di outputs
shutil.copy(csv_path, "last_mile_delivery.csv")

print("Shape:", df.shape)
print("\nKolom:", list(df.columns))
print("\nContoh data:")
display(df.head(3))
print("\nTipe data:")
print(df.dtypes)

In [ ]:
# EDA: ubah data wide → panjang (satu baris per titik pengiriman) lalu rangkum
import pandas as pd
import numpy as np

df = pd.read_csv("last_mile_delivery.csv", sep=";", encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

strata = [f"Jalan {r}" for r in ["I","II","III","IV","V","VI","VII","VIII","IX"]]
suffix = [""] + [f".{i}" for i in range(1, 9)]

def parse_num(v):
    """Angka berformat Indonesia ('164,58') atau teks non-numerik -> (nilai, teks_asli)."""
    if pd.isna(v) or str(v).strip() == "":
        return np.nan, None
    s = str(v).strip().replace(",", ".")
    num = pd.to_numeric(s, errors="coerce")
    return (float(num) if pd.notna(num) else np.nan), (None if pd.notna(num) else s)

records = []
for _, r in df.iterrows():
    kec = r["Kecamatan"]
    for s, sfx in zip(strata, suffix):
        nama = r[s]
        jarak_num, jarak_txt = parse_num(r[f"Jarak (m){sfx}"])
        bagian_num, _ = parse_num(r[f"Bagian{sfx}"])
        ll = r[f"Latitude,Longitude{sfx}"]
        if pd.isna(ll) or str(ll).strip() == "":
            continue
        try:
            lat, lon = [float(x.strip()) for x in str(ll).split(",")]
        except ValueError:
            continue
        records.append({
            "kecamatan": kec, "strata": s, "nama_jalan": nama,
            "jarak_m": jarak_num, "jarak_teks": jarak_txt,
            "bagian": bagian_num,
            "lat": lat, "lon": lon,
        })

pts = pd.DataFrame(records)
print("Titik pengiriman valid:", len(pts), "| Kecamatan:", pts["kecamatan"].nunique())
print("\nTitik per kecamatan:")
print(pts["kecamatan"].value_counts().to_string())
print("\nTitik per strata jalan:")
print(pts["strata"].value_counts().sort_index().to_string())
print("\nRentang koordinat  lat: [%.4f, %.4f]  lon: [%.4f, %.4f]" % (pts.lat.min(), pts.lat.max(), pts.lon.min(), pts.lon.max()))
print("\nJarak dari jalan utama (m) — numerik:")
print(pts["jarak_m"].describe().round(1).to_string())
print("\nNilai teks pada kolom jarak (mis. 'Ujung 1' = ujung jalan):", pts["jarak_teks"].nunique(), "varian unik")
print("Missing: jarak_m =", pts["jarak_m"].isna().sum(), "| bagian =", pts["bagian"].isna().sum())

pts.to_csv("delivery_points_long.csv", index=False)
print("\nSaved -> delivery_points_long.csv")

In [ ]:
# Visualisasi: sebaran geografis titik pengiriman & profil jarak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pts = pd.read_csv("delivery_points_long.csv")
pts["kecamatan"] = pts["kecamatan"].str.replace("\n", "").str.strip()

# Cek nilai teks di kolom Bagian mentah (penanda posisi: Ujung/Tengah jalan)
raw = pd.read_csv("last_mile_delivery.csv", sep=";", encoding="utf-8-sig")
bcols = [c.strip() for c in raw.columns if c.strip().startswith("Bagian")]
teks_bagian = pd.unique(pd.concat([raw[c] for c in bcols]).dropna().astype(str))
teks_bagian = [v for v in teks_bagian if not v.replace(",", "").replace(".", "").replace("-", "").isdigit()]
print("Nilai teks di kolom Bagian:", list(teks_bagian)[:10])

# 1) Peta sebaran titik pengiriman
fig, ax = plt.subplots(figsize=(9, 6.5))
colors = plt.cm.Set2(np.linspace(0, 1, pts["kecamatan"].nunique()))
for (kec, sub), c in zip(pts.groupby("kecamatan"), colors):
    ax.scatter(sub["lon"], sub["lat"], s=14, alpha=0.75, label=kec, color=c, edgecolor="none")
ax.axhline(0.0, color="crimson", lw=1.1, ls="--", alpha=0.85)
ax.text(109.2785, 0.0045, "Garis Khatulistiwa (0°)", color="crimson", fontsize=9)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_title("Sebaran 4.364 Titik Pengiriman — Kota Pontianak (per Kecamatan)")
ax.legend(title="Kecamatan", frameon=False, fontsize=8, markerscale=1.7)
ax.set_facecolor("white"); fig.patch.set_facecolor("white")
plt.tight_layout(); plt.show()

# 2) Jumlah titik per kecamatan + sebaran jarak dari jalan utama
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
vc = pts["kecamatan"].value_counts()
axes[0].barh(vc.index, vc.values, color="#4C72B0")
axes[0].invert_yaxis()
axes[0].set_title("Titik Pengiriman per Kecamatan")
axes[0].set_xlabel("Jumlah titik")
for i, v in enumerate(vc.values):
    axes[0].text(v + 20, i, str(v), va="center", fontsize=9)

j = pts["jarak_m"].dropna()
axes[1].hist(j.clip(upper=1500), bins=40, color="#55A868", edgecolor="white")
axes[1].axvline(j.median(), color="crimson", ls="--", lw=1.2)
axes[1].text(j.median() + 35, axes[1].get_ylim()[1] * 0.9, f"median {j.median():.0f} m",
             color="crimson", fontsize=9)
axes[1].set_title("Jarak Titik dari Jalan Utama (m, cap 1.500 m)")
axes[1].set_xlabel("Jarak (m)")
for a in axes:
    a.set_facecolor("white")
fig.patch.set_facecolor("white")
plt.tight_layout(); plt.show()

## Lampiran — sel fallback lama (Palmer Penguins)

Dua sel di bawah ini adalah eksplorasi *fallback* sementara sebelum dataset asli Indonesia ditemukan (Palmer Penguins dari `seaborn`). Sel tersebut sudah **tidak dipakai** dan hanya disimpan sebagai riwayat; analisis utama notebook ini menggunakan dataset **Last Mile Delivery (Pontianak)** di atas.

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load Palmer Penguins (the user-designated fallback dataset)
penguins = sns.load_dataset("penguins")

print("Shape:", penguins.shape)
print("\nColumns & dtypes:")
print(penguins.dtypes)
print("\nFirst 5 rows:")
display(penguins.head())

print("\nMissing values per column:")
print(penguins.isna().sum())

print("\nDuplicate rows:", penguins.duplicated().sum())

print("\nTarget (species) class balance:")
print(penguins["species"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Numeric summary
print(penguins.describe().T[["mean", "std", "min", "max"]].round(2))

# High-signal view: two most discriminating measurements, colored by species
fig, ax = plt.subplots(figsize=(8, 5))
for sp, color in zip(["Adelie", "Chinstrap", "Gentoo"], ["#4C72B0", "#DD8452", "#55A868"]):
    sub = penguins[penguins["species"] == sp]
    ax.scatter(sub["bill_length_mm"], sub["flipper_length_mm"], label=sp, color=color, alpha=0.75, edgecolor="white", s=40)
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Flipper length (mm)")
ax.set_title("Palmer Penguins — species separate cleanly on measurements")
ax.legend(title="Species", frameon=False)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
plt.tight_layout()
plt.show()

# Export the dataset so you can download it
penguins.to_csv("penguins.csv", index=False)
print("\nSaved -> penguins.csv")

## Kesesuaian Dataset dengan Proposal JalurAI

Proposal **JalurAI** (COMPFEST 18 AIC): sistem prediksi risiko keterlambatan & kelebihan biaya per pesanan (XGBoost classifier + regressor, rule engine, SHAP, Resolver Agent LLM). Bab 3.2 menyebut data historis pesanan (berat, dimensi, jarak tempuh, wilayah asal-tujuan, jenis armada, nilai barang, riwayat keterlambatan rute) dan — karena data distributor riil tertutup — tim berencana menyusun **data sintetis** berpola disparitas Jawa vs luar Jawa 20–40%.

**Penilaian terhadap dataset Last Mile Delivery (Pontianak):**

| Kebutuhan JalurAI | Didukung dataset ini? | Keterangan |
|---|---|---|
| Titik tujuan pengiriman nyata | ✅ Ya | 4.364 koordinat pelanggan riil di 6 kecamatan Pontianak |
| Fitur `jarak_tempuh` | ✅ Sebagian | Bisa dihitung riil (haversine) dari depot → tiap titik, bukan angka acak |
| Proksi aksesibilitas tujuan | ✅ Ya | 9 strata jalan (utama → gang) + jarak dari jalan utama → pengiriman ke gang dalam wajar dimodelkan lebih berisiko |
| Modul peta (GeoPandas/Folium, tahap final) | ✅ Ya | Titik siap divisualisasikan sebagai heatmap risiko per wilayah |
| Atribut pesanan (berat, nilai barang, biaya kirim, armada) | ❌ Tidak ada | Tetap harus disintesis sesuai rencana proposal |
| Label keterlambatan / kelebihan biaya | ❌ Tidak ada | Tetap dari simulasi probabilistik (bimodal + noise) |
| Baseline disparitas Jawa vs luar Jawa | ❌ Tidak | Dataset hanya 1 kota (luar Jawa); baseline internal Jawa tetap dari asumsi/literatur atau data tarif ekspedisi |

**Kesimpulan:** dataset ini *membantu*, tetapi perannya adalah **lapisan geografi nyata untuk mem-grounding data sintetis** — generator pesanan menyampel titik tujuan riil Pontianak sehingga jarak tempuh, wilayah tujuan, dan profil aksesibilitas bersifat nyata, sementara atribut pesanan dan label risiko tetap disimulasikan sesuai desain proposal. Ini menaikkan kredibilitas ("berbasis koordinat pelanggan riil") tanpa mengubah arsitektur pipeline. Dataset ini **bukan** pengganti data historis pesanan. Catatan: lisensi Kaggle bertanda *Unknown* — cukup untuk MVP kompetisi, tetapi hubungi pemilik dataset bila dipakai komersial.

## Dataset Final JalurAI — Pesanan E-Commerce Riil + Label Risiko Tersimulasi

Setelah pencarian lanjutan, ditemukan dataset riil yang jauh lebih cocok untuk JalurAI: **[Indonesia E-Commerce Sales & Shipping 2023–2025](https://www.kaggle.com/datasets/bakitacos/indonesia-e-commerce-sales-and-shipping-20232025)** (CC BY-SA 4.0) — 20.848 pesanan e-commerce Indonesia selama 24 bulan (Des 2023 – Nov 2025) dengan ongkir riil, berat, kota/provinsi tujuan (34 provinsi), opsi kurir, dan status pesanan.

**Strategi:** fitur memakai data riil; hanya **label keterlambatan & kelebihan biaya yang disimulasikan** (data waktu kirim distributor memang tertutup — sesuai rencana Bab 3.2 proposal). Lokasi penjual disimpulkan dari pesanan Same Day/Instant → **Tangerang, Banten**. Jarak tempuh dihitung geodesik (haversine) Tangerang → kota tujuan dari koordinat hasil geocoding.

In [ ]:
# Geocoding kota tujuan (dengan cache) untuk menghitung jarak tempuh riil dari gudang Tangerang
import pandas as pd
import numpy as np
import requests, time, json, os

path = "/root/.cache/kagglehub/datasets/bakitacos/indonesia-e-commerce-sales-and-shipping-20232025/versions/6/all_months_clean.csv"
orders = pd.read_csv(path, sep=";", encoding="utf-8-sig")
print("Pesanan:", orders.shape)

ORIGIN = {"nama": "KOTA TANGERANG", "lat": -6.1783, "lon": 106.6319}

# Fallback koordinat kota besar (dipakai jika geocoding gagal)
FALLBACK = {
    "KOTA JAKARTA BARAT": (-6.1352, 106.8133), "KOTA JAKARTA SELATAN": (-6.2615, 106.8106),
    "KOTA JAKARTA PUSAT": (-6.1865, 106.8343), "KOTA JAKARTA UTARA": (-6.1214, 106.7741),
    "KOTA JAKARTA TIMUR": (-6.2250, 106.9004), "KOTA TANGERANG": (-6.1783, 106.6319),
    "KOTA TANGERANG SELATAN": (-6.2886, 106.7177), "KAB. TANGERANG": (-6.2667, 106.4667),
    "KAB. BOGOR": (-6.5950, 106.8166), "KOTA BOGOR": (-6.5971, 106.8060),
    "KOTA BEKASI": (-6.2383, 106.9756), "KAB. BEKASI": (-6.2417, 107.1083),
    "KOTA DEPOK": (-6.4025, 106.7942), "KAB. BANDUNG": (-6.9175, 107.6191),
    "KOTA BANDUNG": (-6.9175, 107.6191), "KOTA SEMARANG": (-6.9667, 110.4167),
    "KOTA SURABAYA": (-7.2575, 112.7521), "KOTA MEDAN": (3.5952, 98.6722),
    "KOTA PALEMBANG": (-2.9761, 104.7754), "KOTA MAKASSAR": (-5.1477, 119.4327),
    "KOTA DENPASAR": (-8.6705, 115.2126), "KOTA PONTIANAK": (-0.0263, 109.3425),
    "KOTA BALIKPAPAN": (-1.2379, 116.8529), "KOTA PEKANBARU": (0.5071, 101.4478),
    "KOTA PADANG": (-0.9471, 100.4172), "KOTA BANDAR LAMPUNG": (-5.4500, 105.2667),
    "KOTA YOGYAKARTA": (-7.7956, 110.3695), "KOTA MALANG": (-7.9666, 112.6326),
    "KOTA SAMARINDA": (-0.5022, 117.1536), "KOTA BANJARMASIN": (-3.3186, 114.5944),
    "KOTA MANADO": (1.4748, 124.8421), "KOTA KUPANG": (-10.1772, 123.6070),
    "KOTA JAMBI": (-1.6101, 103.6131), "KOTA BENGKULU": (-3.8004, 102.2655),
    "KOTA PALANGKARAYA": (-2.2136, 113.9108), "KOTA MATARAM": (-8.5833, 116.1167),
    "KOTA SERANG": (-6.1100, 106.1639), "KOTA CIREBON": (-6.7320, 108.5523),
    "KOTA SOLO": (-7.5755, 110.8243), "KAB. GARUT": (-7.2279, 107.9087),
    "KAB. SUKABUMI": (-6.9277, 106.9300), "KAB. CIANJUR": (-6.8167, 107.1333),
}

# Koordinat ibu kota provinsi (fallback tingkat 2)
PROV = {
    "ACEH": (5.5483, 95.3238), "SUMATERA UTARA": (3.5952, 98.6722), "SUMATERA BARAT": (-0.9471, 100.4172),
    "RIAU": (0.5071, 101.4478), "KEPULAUAN RIAU": (1.1301, 104.0529), "JAMBI": (-1.6101, 103.6131),
    "SUMATERA SELATAN": (-2.9761, 104.7754), "BENGKULU": (-3.8004, 102.2655), "LAMPUNG": (-5.4500, 105.2667),
    "KEPULAUAN BANGKA BELITUNG": (-2.1316, 106.1169), "DKI JAKARTA": (-6.2088, 106.8456),
    "JAWA BARAT": (-6.9175, 107.6191), "JAWA TENGAH": (-6.9667, 110.4167), "DI YOGYAKARTA": (-7.7956, 110.3695),
    "JAWA TIMUR": (-7.2575, 112.7521), "BANTEN": (-6.1100, 106.1639), "BALI": (-8.6705, 115.2126),
    "NUSA TENGGARA BARAT": (-8.5833, 116.1167), "NUSA TENGGARA TIMUR": (-10.1772, 123.6070),
    "KALIMANTAN BARAT": (-0.0263, 109.3425), "KALIMANTAN TENGAH": (-2.2136, 113.9108),
    "KALIMANTAN SELATAN": (-3.3186, 114.5944), "KALIMANTAN TIMUR": (-0.5022, 117.1536),
    "KALIMANTAN UTARA": (3.0730, 116.0413), "SULAWESI UTARA": (1.4748, 124.8421),
    "SULAWESI TENGAH": (-0.8999, 119.8707), "SULAWESI SELATAN": (-5.1477, 119.4327),
    "SULAWESI TENGGARA": (-3.9985, 122.5130), "GORONTALO": (0.5435, 123.0568),
    "SULAWESI BARAT": (-2.8441, 119.2321), "MALUKU": (-3.6954, 128.1814),
    "MALUKU UTARA": (0.7833, 127.3667), "PAPUA": (-2.5337, 140.7181), "PAPUA BARAT": (-0.8615, 134.0786),
}

CACHE = "city_coords_cache.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

cities = orders[["Kota/Kabupaten", "Provinsi"]].drop_duplicates().dropna()
todo = [r for _, r in cities.iterrows() if r["Kota/Kabupaten"] not in cache]
print(f"Kota unik: {len(cities)} | sudah di-cache: {len(cache)} | perlu geocode: {len(todo)}")

headers = {"User-Agent": "jalurai-research-notebook/1.0 (educational)"}
for i, r in enumerate(todo):
    kota, prov = r["Kota/Kabupaten"], r["Provinsi"]
    try:
        q = f"{kota}, {prov}, Indonesia"
        resp = requests.get("https://nominatim.openstreetmap.org/search",
                            params={"q": q, "format": "json", "limit": 1},
                            headers=headers, timeout=15)
        js = resp.json()
        if js:
            cache[kota] = (float(js[0]["lat"]), float(js[0]["lon"]), "nominatim")
        else:
            cache[kota] = (*PROV.get(prov, (-2.5, 118.0)), "provinsi_fallback")
    except Exception:
        cache[kota] = (*PROV.get(prov, (-2.5, 118.0)), "provinsi_fallback")
    if (i + 1) % 25 == 0:
        print(f"progress {i+1}/{len(todo)}", flush=True)
        json.dump(cache, open(CACHE, "w"))
    time.sleep(1.05)

json.dump(cache, open(CACHE, "w"))
src = pd.Series([v[2] for v in cache.values()]).value_counts()
print("\nSelesai. Sumber koordinat:", src.to_dict())

In [ ]:
# Pass 2 (mandiri): coba lagi kota yang fallback dengan beberapa varian query
import json, requests, time, re
import pandas as pd

orders = pd.read_csv("all_months_clean.csv", sep=";", encoding="utf-8-sig")
kota_prov = orders[["Kota/Kabupaten", "Provinsi"]].drop_duplicates().set_index("Kota/Kabupaten")["Provinsi"].to_dict()

cache = json.load(open("city_coords_cache.json"))
retry = {k: v for k, v in cache.items() if v[2] == "provinsi_fallback"}
print("Kota fallback untuk dicoba ulang:", len(retry))

headers = {"User-Agent": "jalurai-research-notebook/1.0 (educational)"}

def geocode(q):
    r = requests.get("https://nominatim.openstreetmap.org/search",
                     params={"q": q, "format": "json", "limit": 3},
                     headers=headers, timeout=15)
    js = r.json()
    if not js:
        return None
    # pilih hasil terbaik: utamakan city-level (admin level 8/10/12), lalu beri peringkat
    best = None
    for hit in js:
        rank = hit.get("addresstype") in ("city", "town", "village", "municipality", "administrative")
        if best is None or (rank and best[2] != 1):
            best = (float(hit["lat"]), float(hit["lon"]), 1 if rank else 0)
    return best

fixed = 0
for i, kota in enumerate(retry):
    prov = kota_prov.get(kota, "")
    nama_tanpa_prefix = re.sub(r"^(KAB\.|KOTA|KABUPATEN)\s*", "", kota, flags=re.I).title()
    variants = [
        f"{kota.title()}, {prov.title()}, Indonesia",
        f"{nama_tanpa_prefix}, {prov.title()}, Indonesia",
        f"{nama_tanpa_prefix}, Indonesia",
    ]
    ok = None
    for q in variants:
        try:
            ok = geocode(q)
            if ok:
                break
        except Exception:
            continue
        time.sleep(0.4)
    if ok:
        cache[kota] = (ok[0], ok[1], "nominatim")
        fixed += 1
    if (i + 1) % 20 == 0:
        print(f"progress {i+1}/{len(retry)} (fixed={fixed})", flush=True)
        json.dump(cache, open("city_coords_cache.json", "w"))
    time.sleep(1.0)

json.dump(cache, open("city_coords_cache.json", "w"))
src = pd.Series([v[2] for v in cache.values()]).value_counts()
print(f"\nSelesai. Diperbaiki: {fixed} kota. Distribusi akhir:", src.to_dict())

In [ ]:
# Unduh dataset e-commerce Indonesia (versi CLEAN, ML-ready) dan salin ke /home/user
# Sumber: https://www.kaggle.com/datasets/bakitacos/indonesia-e-commerce-sales-and-shipping-20232025 (CC BY-SA 4.0)
import kagglehub, os, shutil, glob

path = kagglehub.dataset_download("bakitacos/indonesia-e-commerce-sales-and-shipping-20232025")
print("Cache:", path)

src = os.path.join(path, "all_months_clean.csv")
dst = "all_months_clean.csv"
shutil.copy(src, dst)
print(f"Disalin -> {dst} ({os.path.getsize(dst)/1e6:.2f} MB)")

In [ ]:
# ==== BANGUN DATASET FINAL JALURAI (v2 — parsing & label dibetulkan) ====
# Fitur & ongkir: RIIL (dataset e-commerce Indonesia, sumber Kaggle bakitacos, CC BY-SA 4.0)
# Label keterlambatan & kelebihan biaya: DISIMULASI (data waktu kirim distributor tertutup,
# sesuai rencana Bab 3.2 proposal; pola mengikuti temuan literatur + disparitas Jawa/luar Jawa)
import pandas as pd
import numpy as np
import json, re

rng = np.random.default_rng(42)  # seed tetap -> reproduksibel

orders = pd.read_csv("all_months_clean.csv", sep=";", encoding="utf-8-sig")
cache = json.load(open("city_coords_cache.json"))
print("Pesanan:", orders.shape[0], "| kota dengan koordinat:", len(cache))

# ---------- 1. Lokasi asal (disimpulkan dari pesanan SameDay/Instant) ----------
ORIGIN = (-6.1783, 106.6319)  # Tangerang, Banten

# ---------- 2. Jarak tempuh geodesik Tangerang -> kota tujuan ----------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1); dl = np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def coord_of(city):
    c = cache.get(city)
    return (c[0], c[1]) if c else (None, None)

out = []
for _, r in orders.iterrows():
    lat, lon = coord_of(r["Kota/Kabupaten"])
    if lat is None:
        continue
    out.append({
        "order_id": r["order_id"], "kota_tujuan": r["Kota/Kabupaten"], "provinsi": r["Provinsi"],
        "lat_tujuan": lat, "lon_tujuan": lon,
        "jarak_tempuh_km": haversine_km(ORIGIN[0], ORIGIN[1], lat, lon),
        "berat_kg": r["total_weight_gr"] / 1000.0, "qty": r["total_qty"],
        "diskon_idr": r["Total Diskon"], "kategori_produk": r["product_categories"],
        "jumlah_kategori": r["num_product_categories"],
        "status_pesanan": r["Status Pesanan"], "opsi_pengiriman": r["Opsi Pengiriman"],
        "metode_pembayaran": r["Metode Pembayaran"],
        "ongkir_dibayar_pembeli": r["Ongkos Kirim Dibayar oleh Pembeli"],
        "estimasi_potongan_idr": r["Estimasi Potongan Biaya Pengiriman"],
        "total_pembayaran_idr": r["Total Pembayaran"],
        "perkiraan_ongkir_idr": r["Perkiraan Ongkos Kirim"],
        "waktu_pesanan": r["Waktu Pesanan Dibuat"],
    })

df = pd.DataFrame(out)
print("Baris dengan koordinat tujuan valid:", len(df), f"({len(df)/len(orders)*100:.1f}%)")

# ---------- 3. Fitur turunan (riil) ----------
t = pd.to_datetime(df["waktu_pesanan"], errors="coerce")
df["waktu_missing"] = t.isna().astype(int)
df["bulan"] = t.dt.month; df["jam"] = t.dt.hour; df["tahun"] = t.dt.year
df["bulan"] = df["bulan"].fillna(df["bulan"].mode()[0]).astype(int)   # imputasi mode
df["jam"] = df["jam"].fillna(df["jam"].mode()[0]).astype(int)

# Normalisasi nama provinsi: "NUSA TENGGARA BARAT (NTB)" -> "NUSA TENGGARA BARAT"
df["provinsi"] = df["provinsi"].str.replace(r"\s*\(.*\)", "", regex=True).str.strip()
df.loc[df["provinsi"] == "NANGGROE ACEH DARUSSALAM", "provinsi"] = "ACEH"

PULAU = {
    "ACEH": "SUMATERA", "SUMATERA UTARA": "SUMATERA", "SUMATERA BARAT": "SUMATERA",
    "RIAU": "SUMATERA", "KEPULAUAN RIAU": "SUMATERA", "JAMBI": "SUMATERA",
    "SUMATERA SELATAN": "SUMATERA", "BENGKULU": "SUMATERA", "LAMPUNG": "SUMATERA",
    "KEPULAUAN BANGKA BELITUNG": "SUMATERA",
    "DKI JAKARTA": "JAWA", "JAWA BARAT": "JAWA", "JAWA TENGAH": "JAWA",
    "DI YOGYAKARTA": "JAWA", "JAWA TIMUR": "JAWA", "BANTEN": "JAWA",
    "BALI": "BALI-NUSA", "NUSA TENGGARA BARAT": "BALI-NUSA", "NUSA TENGGARA TIMUR": "BALI-NUSA",
    "KALIMANTAN BARAT": "KALIMANTAN", "KALIMANTAN TENGAH": "KALIMANTAN",
    "KALIMANTAN SELATAN": "KALIMANTAN", "KALIMANTAN TIMUR": "KALIMANTAN", "KALIMANTAN UTARA": "KALIMANTAN",
    "SULAWESI UTARA": "SULAWESI", "SULAWESI TENGAH": "SULAWESI", "SULAWESI SELATAN": "SULAWESI",
    "SULAWESI TENGGARA": "SULAWESI", "GORONTALO": "SULAWESI", "SULAWESI BARAT": "SULAWESI",
    "MALUKU": "MALUKU-PAPUA", "MALUKU UTARA": "MALUKU-PAPUA", "PAPUA": "MALUKU-PAPUA",
    "PAPUA BARAT": "MALUKU-PAPUA",
}
df["pulau"] = df["provinsi"].map(PULAU).fillna("LAINNYA")
df["luar_jawa"] = (df["pulau"] != "JAWA").astype(int)

# Parsing tier layanan & kurir (robust utk semua format opsi)
KURIR = ["SPX", "J&T", "JNE", "SICEPAT", "NINJA", "ANTERAJA", "POS", "TIKI", "LION",
         "IDEXPRESS", "GOSEND", "GRAB", "J&T"]

def parse_opsi(s):
    s = str(s)
    up = s.upper()
    if "INSTANT" in up: tier = "Instan"
    elif "SAME DAY" in up: tier = "Same Day"
    elif "NEXT DAY" in up: tier = "Next Day"
    elif "HEMAT" in up: tier = "Hemat"
    elif "KARGO" in up or "TRUCKING" in up: tier = "Kargo"
    elif "REGULER" in up: tier = "Reguler"
    else: tier = "Reguler"
    kurir = "LAINNYA"
    for k in KURIR:
        if k in up:
            kurir = k; break
    return tier, kurir

parsed = df["opsi_pengiriman"].apply(parse_opsi)
df["tier_layanan"] = parsed.str[0]; df["kurir"] = parsed.str[1]

# nilai barang & rasio beban distribusi (target utama proposal)
df["nilai_barang_idr"] = (df["total_pembayaran_idr"] - df["ongkir_dibayar_pembeli"]).clip(lower=0)
df["ongkir_total_idr"] = df["ongkir_dibayar_pembeli"] + df["estimasi_potongan_idr"]
df["rasio_beban_distribusi"] = (df["ongkir_total_idr"] / df["nilai_barang_idr"].replace(0, np.nan)).replace([np.inf], np.nan)
df["ongkir_per_kg"] = df["perkiraan_ongkir_idr"] / df["berat_kg"].replace(0, np.nan)
df["rasio_jarak_per_kg"] = df["jarak_tempuh_km"] / df["berat_kg"].replace(0, np.nan)
df["diskon_persen"] = df["diskon_idr"] / df["nilai_barang_idr"].replace(0, np.nan) * 100
df["diskon_ada"] = (df["diskon_idr"] > 0).astype(int)

# indeks disparitas wilayah: median ongkir/kg per provinsi relatif terhadap median Jawa
jawa_prov = ["JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR", "DKI JAKARTA", "DI YOGYAKARTA", "BANTEN"]
med_prov = df.groupby("provinsi")["ongkir_per_kg"].median()
baseline = med_prov[jawa_prov].median()
df["indeks_disparitas_wilayah"] = df["provinsi"].map(med_prov) / baseline

# ---------- 4. Simulasi label keterlambatan (probabilistik, seed tetap) ----------
df["berat_2_4kg"] = df["berat_kg"].between(2, 4, inclusive="left").astype(int)
df["bulan_peak"] = df["bulan"].isin([11, 12, 1]).astype(int)
df["cod"] = df["metode_pembayaran"].str.contains("COD", na=False).astype(int)

tier_bonus = {"Instan": -1.2, "Same Day": -0.9, "Next Day": -0.6, "Reguler": -0.25,
              "Hemat": 0.35, "Kargo": 0.55}
logit = (-1.95
         + 0.65 * df["luar_jawa"]
         + 0.55 * np.log1p(df["jarak_tempuh_km"]) / 5.5
         + 0.55 * df["berat_2_4kg"]          # temuan: berat 2-4 kg cenderung telat
         + 0.30 * df["diskon_ada"]           # ada diskon -> margin tipis, cenderung telat
         + df["tier_layanan"].map(tier_bonus).fillna(0.0)
         + 0.30 * df["bulan_peak"]
         + 0.25 * df["cod"]
         + rng.normal(0, 0.40, len(df)))     # noise Gaussian per rencana proposal
p_late = 1 / (1 + np.exp(-logit))
df["prob_telat"] = p_late
df["label_telat"] = rng.binomial(1, p_late)

# hari keterlambatan (hanya untuk yang telat): gamma dgn skala tergantung jarak
lag = rng.gamma(shape=1.6, scale=0.9, size=len(df)) + 0.25 * np.log1p(df["jarak_tempuh_km"])
df["hari_keterlambatan"] = np.where(df["label_telat"] == 1, lag.clip(1, 14).round(1), 0.0)

# ---------- 5. Simulasi kelebihan biaya (cost overrun) ----------
logit2 = (-1.55
          + 0.55 * df["luar_jawa"]
          + 0.35 * df["berat_2_4kg"]
          + 0.40 * df["label_telat"]
          + 0.30 * (df["ongkir_per_kg"] > df["ongkir_per_kg"].median()).astype(int)
          + rng.normal(0, 0.35, len(df)))
p_over = 1 / (1 + np.exp(-logit2))
df["label_kelebihan_biaya"] = rng.binomial(1, p_over)
overrun_pct = rng.uniform(0.08, 0.25, len(df))  # 8-25% dari ongkir
df["nilai_kelebihan_biaya_idr"] = np.where(df["label_kelebihan_biaya"] == 1,
                                           (df["ongkir_total_idr"] * overrun_pct).round(0), 0.0)

# ---------- 6. Pesanan batal = tidak dikirim -> bukan objek risiko keterlambatan ----------
batal = df["status_pesanan"].str.contains("Batal", na=False)
for c in ["label_telat", "prob_telat", "hari_keterlambatan", "label_kelebihan_biaya", "nilai_kelebihan_biaya_idr"]:
    df.loc[batal, c] = 0
df["is_cancelled"] = batal.astype(int)
print("Pesanan batal di-nolkan labelnya:", int(batal.sum()))

# ---------- 7. Rapi & simpan ----------
cols = ["order_id", "waktu_pesanan", "tahun", "bulan", "jam", "waktu_missing",
        "status_pesanan", "is_cancelled",
        "kategori_produk", "jumlah_kategori", "qty", "berat_kg", "berat_2_4kg",
        "kota_tujuan", "provinsi", "pulau", "luar_jawa",
        "tier_layanan", "kurir", "metode_pembayaran",
        "diskon_idr", "diskon_persen", "diskon_ada",
        "nilai_barang_idr", "ongkir_dibayar_pembeli", "estimasi_potongan_idr",
        "ongkir_total_idr", "total_pembayaran_idr", "perkiraan_ongkir_idr",
        "jarak_tempuh_km", "ongkir_per_kg", "rasio_jarak_per_kg",
        "indeks_disparitas_wilayah", "rasio_beban_distribusi",
        "prob_telat", "label_telat", "hari_keterlambatan",
        "label_kelebihan_biaya", "nilai_kelebihan_biaya_idr",
        "lat_tujuan", "lon_tujuan"]
df[cols].to_csv("jalurai_orders.csv", index=False)
print("Saved -> jalurai_orders.csv |", df.shape[0], "baris x", len(cols), "kolom")

In [ ]:
# ===== VALIDASI DATASET FINAL JALURAI =====
import pandas as pd
import numpy as np

df = pd.read_csv("jalurai_orders.csv")
print("Shape:", df.shape)

print("\n--- Label balance (hanya pesanan dikirim, is_cancelled=0) ---")
aktif = df[df["is_cancelled"] == 0]
print("label_telat:", aktif["label_telat"].value_counts().to_dict(), f"({aktif['label_telat'].mean()*100:.1f}% telat)")
print("label_kelebihan_biaya:", aktif["label_kelebihan_biaya"].value_counts().to_dict(), f"({aktif['label_kelebihan_biaya'].mean()*100:.1f}%)")

print("\n--- Target 'rasio beban distribusi' (biaya kirim / nilai barang) ---")
print(df["rasio_beban_distribusi"].describe().round(3).to_string())

print("\n--- Jarak tempuh & disparitas ---")
print("jarak_tempuh_km:", df["jarak_tempuh_km"].describe().round(1).to_string().replace("\n", " | "))
print("\nMedian ongkir/kg per pulau (IDR):")
print(df.groupby("pulau")["ongkir_per_kg"].median().round(0).to_string())
print("\nIndeks disparitas wilayah (median per provinsi vs baseline Jawa):")
print(df.groupby("provinsi")["indeks_disparitas_wilayah"].median().sort_values(ascending=False).head(8).round(2).to_string())

print("\n--- Sanity check: tingkat telat per segmen (pola harus masuk akal) ---")
print("luar_jawa :", df.groupby("luar_jawa")["label_telat"].mean().round(3).to_dict())
print("tier      :", df.groupby("tier_layanan")["label_telat"].mean().round(3).to_dict())
print("berat_2_4 :", df.groupby("berat_2_4kg")["label_telat"].mean().round(3).to_dict())
print("diskon_ada:", df.groupby("diskon_ada")["label_telat"].mean().round(3).to_dict())

print("\n--- Missing (dokumentasi) ---")
miss = df.isna().sum()
print(miss[miss > 0].to_string() if (miss > 0).any() else "tidak ada missing")
print("\nContoh 3 baris:")
display(df.head(3))

In [ ]:
# ===== VISUALISASI: pola risiko & disparitas di dataset final =====
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("jalurai_orders.csv")
aktif = df[df["is_cancelled"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

# Panel kiri: tingkat keterlambatan per tier layanan
tier_rate = aktif.groupby("tier_layanan")["label_telat"].mean().sort_values(ascending=False)
colors = plt.cm.YlOrRd(np.linspace(0.25, 0.95, len(tier_rate)))
b = axes[0].bar(tier_rate.index, tier_rate.values * 100, color=colors, edgecolor="white")
axes[0].set_ylabel("Tingkat keterlambatan (%)")
axes[0].set_title("Risiko Keterlambatan per Tier Layanan (simulasi)")
for r, v in zip(b, tier_rate.values * 100):
    axes[0].text(r.get_x() + r.get_width()/2, v + 0.8, f"{v:.1f}%", ha="center", fontsize=9)

# Panel kanan: Jawa vs luar Jawa
jv = aktif.groupby("luar_jawa")["label_telat"].mean() * 100
b2 = axes[1].bar(["Jawa", "Luar Jawa"], jv.values, color=["#4C72B0", "#C44E52"], edgecolor="white", width=0.5)
for r, v in zip(b2, jv.values):
    axes[1].text(r.get_x() + r.get_width()/2, v + 1, f"{v:.1f}%", ha="center", fontsize=10)
axes[1].set_ylabel("Tingkat keterlambatan (%)")
axes[1].set_title("Disparitas Jawa vs Luar Jawa (simulasi)")
axes[1].set_ylim(0, max(jv.values) * 1.25)
for a in axes:
    a.set_facecolor("white"); a.grid(axis="y", alpha=0.25)
fig.patch.set_facecolor("white")
plt.tight_layout(); plt.show()

# Figure 2: ongkir per kg per pulau (disparitas riil dari data)
fig, ax = plt.subplots(figsize=(8, 4.4))
ongkir_pulau = df.groupby("pulau")["ongkir_per_kg"].median().sort_values()
colors2 = ["#C44E52" if p != "JAWA" else "#4C72B0" for p in ongkir_pulau.index]
b = ax.barh(ongkir_pulau.index, ongkir_pulau.values, color=colors2, edgecolor="white")
for r, v in zip(b, ongkir_pulau.values):
    ax.text(v + 400, r.get_y() + r.get_height()/2, f"Rp {v:,.0f}/kg", va="center", fontsize=9)
ax.set_xlabel("Median ongkos kirim per kg (IDR)")
ax.set_title("Disparitas Ongkir Riil per Pulau (dari data e-commerce)")
ax.set_facecolor("white"); ax.grid(axis="x", alpha=0.25)
fig.patch.set_facecolor("white")
plt.tight_layout(); plt.show()

## Kamus Data — `jalurai_orders.csv` (20.848 pesanan × 41 kolom)

**Sumber fitur:** riil — *Indonesia E-Commerce Sales & Shipping 2023–2025* (Kaggle, CC BY-SA 4.0), 24 bulan (Des 2023 – Nov 2025), penjual di Tangerang (disimpulkan dari pesanan Same Day/Instant). **Label keterlambatan & kelebihan biaya: simulasi ber-seed (42)** karena data waktu kirim distributor tertutup (sesuai rencana Bab 3.2 proposal).

| Kelompok | Kolom | Keterangan |
|---|---|---|
| Identitas | `order_id`, `waktu_pesanan`, `tahun`, `bulan`, `jam`, `waktu_missing` | Timestamp pesanan; 1.980 baris tanpa waktu (diimputasi mode + flag) |
| Status | `status_pesanan`, `is_cancelled` | 2.830 pesanan batal → label risiko dinolkan (tidak dikirim) |
| Produk | `kategori_produk`, `jumlah_kategori`, `qty`, `berat_kg`, `berat_2_4kg` | Berat 2–4 kg = segmen berisiko (temuan literatur) |
| Geografi | `kota_tujuan`, `provinsi`, `pulau`, `luar_jawa`, `lat_tujuan`, `lon_tujuan` | 424 kota, 34 provinsi; koordinat dari geocoding (418 presisi, 6 ibu kota provinsi) |
| Pengiriman | `tier_layanan` (Instan/Same Day/Next Day/Reguler/Hemat/Kargo), `kurir`, `metode_pembayaran` | Parsing dari `Opsi Pengiriman` |
| Biaya | `diskon_idr`, `diskon_persen`, `diskon_ada`, `nilai_barang_idr`, `ongkir_dibayar_pembeli`, `estimasi_potongan_idr`, `ongkir_total_idr`, `total_pembayaran_idr`, `perkiraan_ongkir_idr` | Ongkir riil dari marketplace |
| Fitur turunan | `jarak_tempuh_km` (haversine Tangerang→tujuan), `ongkir_per_kg`, `rasio_jarak_per_kg`, `indeks_disparitas_wilayah`, `rasio_beban_distribusi` | `indeks_disparitas` = median ongkir/kg provinsi ÷ baseline Jawa (NTT 4,97×, Papua Barat 3,97×) |
| Target simulasi | `prob_telat`, `label_telat`, `hari_keterlambatan`, `label_kelebihan_biaya`, `nilai_kelebihan_biaya_idr` | 30,1% telat pada pesanan aktif; pola: luar Jawa 39,3% vs Jawa 22,6%; Instan 4,6% < Reguler 18,1% < Kargo 29,7% |

**Missing yang wajar:** `rasio_beban_distribusi` & `diskon_persen` NaN (2.893 baris) = nilai barang Rp 0 (pesanan batal/refund); filter `is_cancelled==0` untuk modeling keterlambatan.

**Cara pakai:** target klasifikasi = `label_telat`; target regresi = `hari_keterlambatan`; target biaya = `label_kelebihan_biaya` / `nilai_kelebihan_biaya_idr`; metrik utama proposal = `rasio_beban_distribusi`.